In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms.functional as func

from torchvision.models import resnet18

class ResNet18Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet18(weights=None)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool = resnet.maxpool
        self.encoder1 = resnet.layer1
        self.encoder2 = resnet.layer2
        self.encoder3 = resnet.layer3
        self.encoder4 = resnet.layer4
    def forward(self, x):
        x0 = self.stem(x)
        x1 = self.encoder1(self.pool(x0))
        x2 = self.encoder2(x1)
        x3 = self.encoder3(x2)
        x4 = self.encoder4(x3)
        return x0, x1, x2, x3, x4


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(skip_channels + out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        skip = func.center_crop(skip, [x.shape[2], x.shape[3]])
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class ResNetUNet(nn.Module):
    def __init__(self, n_classes=1):
        super().__init__()
        self.encoder = ResNet18Encoder()

        self.decoder4 = DecoderBlock(512, 256, 256)
        self.decoder3 = DecoderBlock(256, 128, 128)
        self.decoder2 = DecoderBlock(128, 64, 64)
        self.decoder1 = DecoderBlock(64, 64, 32)

        self.final_conv = nn.Conv2d(32, n_classes, kernel_size=1)

    def forward(self, x):
        x0, x1, x2, x3, x4 = self.encoder(x)

        d4 = self.decoder4(x4, x3)
        d3 = self.decoder3(d4, x2)
        d2 = self.decoder2(d3, x1)
        d1 = self.decoder1(d2, x0)

        out = self.final_conv(d1)
        return out

In [ ]:
from torch.utils.data import Dataset
import tifffile as tif
import random
from torchvision import transforms
import os

random.seed(42)

imageTransform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
maskTransform = transforms.Compose([transforms.ToTensor()])

class LandslideDataset(Dataset):
    def __init__(self, img_list, mask_list):
        self.img_list = img_list
        self.mask_list = mask_list
    
    def __len__(self):
        return len(self.img_list)
    
    def __getitem__(self, index):
        img_dir = self.img_list[index]
        mask_dir = self.mask_list[index]

        img = tif.imread(img_dir)
        mask = tif.imread(mask_dir)

        img = imageTransform(img)
        mask = (maskTransform(mask) > 0).float()

        return img, mask

path = "../../data/"
limit = 100
regions = [f for f in os.listdir(path) if (os.path.isdir(os.path.join(path, f)) and f != "study areas shp")]

regions_dict = dict()
regions_pos_weight = dict()

for region in regions:
    dataset_dir = path + region
    image_dir = os.path.join(dataset_dir, "img")
    img_list = os.listdir(image_dir)

    all_images = sorted(os.path.join(image_dir, f) for f in img_list)

    if len(all_images) > limit:
        all_images = random.sample(all_images, limit)

    regions_dict[region] = all_images
    regions_pos_weight[region] = 0

def estimate_pos_weight(mask_paths):
    pos, tot = 0, 0
    for path in mask_paths:
        m = tif.imread(path)
        m = torch.from_numpy((m > 0).astype("float32"))
        pos += m.sum().item()
        tot += m.numel()
    neg = max(tot - pos, 1.0)
    pos = max(pos, 1.0)
    return neg / pos

# for region in regions_dict:
#     masks = [f.replace("img", "mask") for f in regions_dict[region]]
#    
#     regions_pos_weight[region] = estimate_pos_weight(masks)

In [3]:
import numpy as np
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### TUNE THIS CONSTANT
threshold = 0.6

def getMetrics(TP, TN, FP, FN):
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
        
    iou_1  = TP / (TP + FP + FN + 1e-8)
    iou_0 = TN / (TN + FP + FN + 1e-8)
    miou = (iou_0 + iou_1) / 2 
        
    oa = (TP + TN)/(TP + TN + FP + FN + 1e-8)
    
    return precision, recall, f1, iou_1, miou, oa

def dice_loss(pred, target, smooth=1.):
    pred = torch.sigmoid(pred)
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum()
    return 1 - ((2. * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth))

def boundary_f1_score(pred_mask, true_mask, tolerance=3):
    if isinstance(pred_mask, torch.Tensor):
        pred = pred_mask.detach().cpu().float()
    else:
        pred = torch.from_numpy(pred_mask).float()

    if isinstance(true_mask, torch.Tensor):
        true = true_mask.detach().cpu().float()
    else:
        true = torch.from_numpy(true_mask).float()

    pred = (pred > threshold).float()
    true = (true > threshold).float()

    def _to_bchw(m):
        if m.ndim == 2:
            m = m.unsqueeze(0).unsqueeze(0)
        elif m.ndim == 3:
            m = m.unsqueeze(0)
        return m

    pred = _to_bchw(pred)
    true = _to_bchw(true)

    def get_edges(mask):
        dil = F.max_pool2d(mask, kernel_size=3, stride=1, padding=1)
        ero = 1.0 - F.max_pool2d(1.0 - mask, kernel_size=3, stride=1, padding=1)
        edge = (dil - ero > 0).float()
        return edge

    edge_pred = get_edges(pred)
    edge_true = get_edges(true)

    if edge_pred.sum() == 0 and edge_true.sum() == 0:
        return 1.0
    if edge_pred.sum() == 0 or edge_true.sum() == 0:
        return 0.0

    if tolerance > 0:
        k = 2 * tolerance + 1
        pad = tolerance
        edge_true_dil = F.max_pool2d(edge_true, kernel_size=k, stride=1, padding=pad)
        edge_pred_dil = F.max_pool2d(edge_pred, kernel_size=k, stride=1, padding=pad)
    else:
        edge_true_dil = edge_true
        edge_pred_dil = edge_pred

    precision_b = (edge_pred * edge_true_dil).sum() / (edge_pred.sum() + 1e-8)
    recall_b = (edge_true * edge_pred_dil).sum() / (edge_true.sum() + 1e-8)

    if precision_b + recall_b == 0:
        return 0.0

    bf1 = 2.0 * precision_b * recall_b / (precision_b + recall_b)
    return bf1.item()

### TUNE THESE CONSTANTS
pos_weight = 4.5
lambda_dice = 1.0

criterion_ce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))

def seg_loss(logits, targets):
    return criterion_ce(logits, targets) + lambda_dice * dice_loss(logits, targets)

In [4]:
def train_model(model, optimizer, train_loader, val_loader, epochs, model_path, region_name):
    best_iou = 0.0
    patience = 10
    counter = 0

    print(f'region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa, bf1')

    for epoch in range(epochs):
        model.train()
        running_loss, n_samples = 0.0, 0

        for images, masks in train_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).float()

            optimizer.zero_grad(set_to_none=True)

            logits = model(images)
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

            loss = seg_loss(logits, masks)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * masks.numel()
            n_samples += masks.numel()
        
        training_loss = running_loss / max(1, n_samples)

        model.eval()
        val_loss, val_num = 0.0, 0
        
        TP, FP, FN, TN = 0,0,0,0

        bf1_sum = 0.0
        bf1_count = 0

        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True).float()

                logits = model(images)
                if logits.shape[-2:] != masks.shape[-2:]:
                    logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

                loss = seg_loss(logits, masks)
                val_loss += loss.item()

                preds = torch.sigmoid(logits) > threshold

                for i in range(preds.size(0)):
                    bf1 = boundary_f1_score(preds[i, 0], masks[i, 0], tolerance=3)
                    bf1_sum += bf1
                    bf1_count += 1

                preds_flat = preds.view(-1)
                mask_flat = masks.view(-1)

                TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
                FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
                FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
                TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
                
                val_num += 1
             
        avg_bf1 = bf1_sum / max(bf1_count, 1)
        precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)

        print(f'{region_name}, {epoch+1}, {training_loss :.3f}, {val_loss / val_num :.3f}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}, {avg_bf1:.4f}')

        if epoch == 0:
            torch.save(model.state_dict(), model_path)

        if iou > best_iou:
            best_iou = iou
            counter = 0
            
            torch.save(model.state_dict(), model_path)
        elif iou != 0.0:
            counter += 1
            if counter >= patience:
                break

def test_model(model, test_loader):
    TP, FP, FN, TN = 0,0,0,0

    model.eval()

    bf1_sum = 0.0
    bf1_count = 0

    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).float()

            logits = model(images)
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

            preds = torch.sigmoid(logits) > threshold

            for i in range(preds.size(0)):
                bf1 = boundary_f1_score(preds[i, 0], masks[i, 0], tolerance=3)
                bf1_sum += bf1
                bf1_count += 1

            preds_flat = preds.view(-1)
            mask_flat = masks.view(-1)

            TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
            FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
            FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
            TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()

    avg_bf1 = bf1_sum / max(bf1_count, 1)
    precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)

    return precision, recall, f1, iou, miou, oa, avg_bf1

In [6]:
""" SPN METHODS """

import numpy as np
import cv2
from numpy.fft import fft2, fftshift, ifft2, ifftshift

def load_gray(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return img.astype(np.float32) / 255.0

def radial_bins(h, w, K=64):
    yy, xx = np.ogrid[:h, :w]
    r = np.hypot(yy - h/2, xx - w/2)
    rmax = r.max()
    edges = np.linspace(0, rmax + 1e-6, K + 1)
    return r, edges

def avg_radial_power(images, K=64):
    h, w = images[0].shape
    r, edges = radial_bins(h, w, K)
    P = np.zeros(K, dtype=np.float64)

    for im in images:
        F = fftshift(fft2(im))
        power = np.abs(F) ** 2
        for k in range(K):
            mask = (r >= edges[k]) & (r < edges[k+1])
            if np.any(mask):
                P[k] += power[mask].mean()

    P /= len(images)
    P = cv2.GaussianBlur(P.reshape(-1,1), (5,1), 0).reshape(-1)
    P = P / (P.max() + 1e-8)
    return P, edges


### TUNE gmin and gmax, taper_bins
def spn(image, P_target, edges, gmin=0.6, gmax=1.7, taper_bins=2):
    h, w = image.shape
    K = len(P_target)
    F = fftshift(fft2(image))
    A, ph = np.abs(F), np.angle(F)

    r, _ = radial_bins(h, w, K)
    P_src = np.zeros(K, dtype=np.float64)
    for k in range(K):
        mask = (r >= edges[k]) & (r < edges[k+1])
        if np.any(mask):
            P_src[k] = (A[mask]**2).mean()

    P_src = P_src / (P_src.max() + 1e-8)

    g = np.sqrt((P_target + 1e-8) / (P_src + 1e-8))

    ### print to check this value before clipping
    g = np.clip(g, gmin, gmax)
    
    def cosine_taper(arr, n):
        arr = arr.copy()
        for i in range(n):
            w = 0.5 - 0.5*np.cos(np.pi*(i+1)/(n+1))
            arr[i] = 1.0 + (arr[i]-1.0)*w
            arr[-(i+1)] = 1.0 + (arr[-(i+1)]-1.0)*w
        return arr
    
    ### print to check this value before tapering
    g = cosine_taper(g, taper_bins)

    A2 = np.zeros_like(A)
    for k in range(K):
        mask = (r >= edges[k]) & (r < edges[k+1])
        A2[mask] = A[mask] * g[k]

    F2 = A2 * np.exp(1j * ph)
    out = np.real(ifft2(ifftshift(F2)))

    ### print min and max to check if clipping distorts too much
    return np.clip(out, 0.0, 1.0)

In [7]:
target_spn = dict()

for region in regions_dict:
    tgt_imgs = [load_gray(p) for p in regions_dict[region]]
    P_t, edges = avg_radial_power(tgt_imgs, K=64)
    target_spn[region] = (P_t, edges)
    
np.save("../../spn_profiles/target_params/target_spn.npy", target_spn, allow_pickle=True)

In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import torch.optim as optim

batch_size = 16

output = "source_region,target_region,precision,recall,f1,iou,miou,oa,bf1"

target_spn = np.load("../../spn_profiles/target_params/target_spn.npy", allow_pickle=True).item()

for source_region in regions_dict:
    source_images = regions_dict[source_region]
    source_masks = [f.replace("img", "mask") for f in source_images]

    train_img, val_img, train_mask, val_mask = train_test_split(
        source_images, source_masks, test_size=.2, random_state=42
    )

    for target_region in regions_dict:
        if source_region != target_region:
            epochs = 40
            model = ResNetUNet().to(device)

            P_t, edges = target_spn[target_region]

            out_dir = os.path.join("../../spn_profiles/spn_converted", source_region, target_region)
            os.makedirs(out_dir, exist_ok=True)

            train_spn_imgs = []
            for img_path in train_img:
                src = load_gray(img_path)
                src_spn = spn(src, P_t, edges)
                
                filename = os.path.basename(img_path)
                out_path = os.path.join(out_dir, filename)
                cv2.imwrite(out_path, cv2.cvtColor((src_spn * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB))

                train_spn_imgs.append(out_path)

            train_dataset = LandslideDataset(train_spn_imgs, train_mask)
            val_dataset = LandslideDataset(val_img, val_mask)
            test_dataset = LandslideDataset(
                regions_dict[target_region],
                [f.replace("img", "mask") for f in regions_dict[target_region]],
            )            

            trainLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
            valLoader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
            testLoader = DataLoader(test_dataset, batch_size=batch_size,shuffle=False,num_workers=0)

            optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

            model_path = f"../../models/spn/{source_region}_{target_region}.pth"

            train_model(model, optimizer, trainLoader, valLoader, epochs, model_path, source_region)

            model.load_state_dict(torch.load(model_path, map_location=device))

            precision, recall, f1, iou, miou, oa, bf1 = test_model(model, testLoader)
            result = f'{source_region}, {target_region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}, {bf1:.4f}'
            print(result)
            output += f'\n{result}'

with open(f"../../results/spn/test1.csv", "w") as f:
    f.write(output)


region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa, bf1
Hokkaido Iburi-Tobu, 1, 1.787, 1.742, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 2, 1.764, 1.732, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 3, 1.732, 1.707, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 4, 1.691, 1.639, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 5, 1.616, 1.452, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 6, 1.527, 4.397, 0.0000, 0.0000, 0.0000, 0.0000, 0.4684, 0.9367, 0.0000
Hokkaido Iburi-Tobu, 7, 1.441, 3.608, 0.0002, 0.0000, 0.0000, 0.0000, 0.4673, 0.9345, 0.0009
Hokkaido Iburi-Tobu, 8, 1.349, 2.176, 0.3999, 0.0124, 0.0240, 0.0122, 0.4744, 0.9368, 0.0194
Hokkaido Iburi-Tobu, 9, 1.296, 1.448, 0.5208, 0.1473, 0.2297, 0.1297, 0.5335, 0.9379, 0.1084
Hokkaido Iburi-Tobu, 10, 1.331, 1.304, 0.3896, 0.3538, 0.3708, 0.2276, 0.5752, 0.924

: 